# 01 — Data Preparation

**Sentimentanalys av flygbolagstweets — steg 1/6**

Den här notebooken hämtar rådatan, städar den och sparar tränings-/validerings-/testset
till disk så att resten av pipelinen kan bygga vidare på exakt samma data.

## Om den här pipelinen

Det här projektet är uppdelat i 6 notebooks som körs i ordning — varje notebook gör
**en sak** och sparar resultatet till disk så nästa notebook kan bygga vidare, precis som i
Biometric Access Terminal-projektet:

| # | Notebook | Vad den gör |
|---|----------|--------------|
| 01 | `01_data_preparation.ipynb` | Hämtar, städar och delar upp datan |
| 02 | `02_eda.ipynb` | Utforskar datan och bygger databerättelsen |
| 03 | `03_unsupervised_clustering.ipynb` | Undersöker om förtränade BERT-embeddings själva grupperar tweets efter sentiment — *innan* någon träning |
| 04 | `04_supervised_classification.ipynb` | Transfer learning: tränar DistilBERT att klassificera sentiment |
| 05 | `05_negative_reason_classification.ipynb` | Transfer learning på en andra uppgift: klassificera *varför* en tweet är negativ |
| 06 | `06_evaluation.ipynb` | Utvärderar båda modellerna och sammanfattar resultatet |

> **Kör alla 6 i samma Colab-session/runtime, i ordning.** Varje notebook läser filer som
> en tidigare notebook sparat till `./data/`, `./charts/` eller `./models/`. Om du öppnar en
> ny Colab-runtime mellan notebooks försvinner filerna — montera då Google Drive, eller kör
> allt i en och samma session.

**Varför notebook 05 heter något annat än i Biometric Access Terminal:** där hade ni en
`05_age_gender_estimation.ipynb` som en andra, kompletterande klassificeringsuppgift utifån
samma bilddata. För vårt textdataset finns ingen ålder/kön att skatta — så vi återanvänder
samma *roll* i pipelinen (en andra, kompletterande transfer learning-uppgift på samma rådata)
men applicerar den på något som faktiskt finns i vår data: att klassificera **orsaken** till en
negativ tweet (försening, kundservice, borttappat bagage, osv).



## Imports & mappar

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

os.makedirs("data", exist_ok=True)
print("Mappen ./data/ är redo.")


## Hämta rådata

**Dataset:** [Twitter US Airline Sentiment](https://www.kaggle.com/crowdflower/twitter-airline-sentiment)
(Kaggle / Crowdflower "Data for Everyone"). 14 640 tweets om sex amerikanska flygbolag från
februari 2015, redan manuellt taggade med sentiment **och** — för negativa tweets — en
anledning (`negativereason`), vilket vi återanvänder i notebook 05.

Vi laddar den från en publik GitHub-spegling så att notebooken går att köra utan
Kaggle-inloggning.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/satyajeetkrjha/kaggle-Twitter-US-Airline-Sentiment-/master/Tweets.csv"

raw = pd.read_csv(DATA_URL)
print("Rader, kolumner:", raw.shape)
raw.head(3)


**Vad output visar:** `(14640, 15)` betyder 14 640 tweets och 15 kolumner i rådatan.
Tabellen ovanför visar de tre första raderna så vi kan se hur kolumnerna ser ut — bland annat
`text` (själva tweeten), `airline_sentiment` (facit) och `negativereason` (bara ifylld för
negativa tweets). Vi kommer bara behöva en delmängd av dessa 15 kolumner.


## Städa data

In [ ]:
df = raw[["text", "airline_sentiment", "airline", "negativereason"]].copy()

# Ta bort rader utan text eller sentiment-etikett
df = df.dropna(subset=["text", "airline_sentiment"]).reset_index(drop=True)

# Enkel textstädning: trimma whitespace
df["text"] = df["text"].str.strip()

print("Efter städning:", df.shape)
print()
print(df["airline_sentiment"].value_counts())
print()
print("Saknade värden kvar:")
print(df[["text", "airline_sentiment"]].isna().sum())


**Vad output visar:** Formen efter städning ska fortfarande vara nära `(14640, 4)` — vi
tappar bort några enstaka rader om det finns tomma värden. `value_counts()` visar hur många
tweets som är negativa/neutrala/positiva i hela datasetet: **negative är klart störst**
(runt 9 200), vilket är en viktig obalans att komma ihåg när vi senare tolkar modellens
resultat. `isna().sum()` ska visa **0** för båda kolumnerna — annars har städningen missat
något.


## Spara hela det städade datasetet

Den fulla, städade datan (14 640 rader) sparas separat — den används av **notebook 02 (EDA)**
för databerättelsen, och av **notebook 05** som hämtar sitt eget urval av negativa tweets
direkt härifrån (eftersom det större datasetet ger bättre klassbalans för den uppgiften).


In [ ]:
df.to_csv("data/full_clean.csv", index=False)
print("Sparad: data/full_clean.csv", df.shape)


**Vad output visar:** en bekräftelse på att filen sparats, plus dess dimensioner. Om du
inte ser den här filen i filpanelen till vänster i Colab efteråt — kontrollera att cellen
verkligen kördes (ingen felmeddelande) och att du tittar i rätt mapp (`data/` under den
mapp notebooken körs i).


## Stratifierat urval för snabb träning

Hela datasetet fungerar utmärkt, men för att hålla notebook 03–04 **korta & snabba** tar vi
ett stratifierat urval på 3 600 tweets (samma sentiment-fördelning som originalet bevaras)
och delar upp i train / validation / test.


In [ ]:
SAMPLE_SIZE = 3600

sample, _ = train_test_split(
    df, train_size=SAMPLE_SIZE, stratify=df["airline_sentiment"], random_state=RANDOM_STATE
)

train_val, test_df = train_test_split(
    sample, test_size=0.15, stratify=sample["airline_sentiment"], random_state=RANDOM_STATE
)
train_df, val_df = train_test_split(
    train_val, test_size=0.176, stratify=train_val["airline_sentiment"], random_state=RANDOM_STATE
)  # ca 70/15/15 av totalen

print("Train:", train_df.shape, " Val:", val_df.shape, " Test:", test_df.shape)
print()
print(train_df["airline_sentiment"].value_counts(normalize=True).round(2))


**Vad output visar:** train/val/test ska vara ungefär **2520 / 540 / 540** rader (70/15/15%
av de 3 600 utvalda). Den sista utskriften visar **andelen** negative/neutral/positive i
träningssetet — den ska ligga mycket nära 0.63 / 0.21 / 0.16, alltså samma fördelning som i
hela datasetet. Det är poängen med `stratify=` — annars hade en slumpmässig delning kunnat
råka ge för få exempel av en klass i t.ex. testsetet.


In [ ]:
train_df.to_csv("data/train.csv", index=False)
val_df.to_csv("data/val.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

print("Sparade filer i ./data/:")
for f in ["full_clean.csv", "train.csv", "val.csv", "test.csv"]:
    path = f"data/{f}"
    n = sum(1 for _ in open(path)) - 1
    print(f"  {f:16s} {n} rader")


**Vad output visar:** en lista med de fyra sparade filerna och antal rader i var och en
— ett snabbt sanity-check att allt sparades korrekt innan vi går vidare till nästa notebook.


## Sammanfattning

Vi har nu:
- `data/full_clean.csv` — hela det städade datasetet (14 640 rader), för EDA och notebook 05.
- `data/train.csv`, `data/val.csv`, `data/test.csv` — ett stratifierat urval (3 600 rader
  totalt) för snabb träning av sentimentmodellen i notebook 03–04.

**Nästa steg:** öppna `02_eda.ipynb`.
